# 01.05 章节实践：独立完成 VectorAdd

## 小节概述

本节是一项综合编程实践。你将补全三个学生接口，把 01.04 中的数据切分推导和 01.03 中的 <code>CopyIn → Compute → CopyOut</code> 数据通路落实为可以在 NPU 上运行的 VectorAdd 算子。

<strong>实践内容：</strong> 建立工作副本，理解三个 TODO 的调用位置，完成核间偏移、核内偏移和 Vector 加法，并通过构建、非法参数与精度自检确认结果。

- <strong>建议用时：</strong> 35 分钟；
- <strong>前置要求：</strong> 已经完成 [01.02 环境与工程一键启动](01.02_environment_and_project.ipynb)、[01.03 VectorAdd 算子实验](01.03_vector_add_operator.ipynb) 和 [01.04 Tiling 可视化交互实验](01.04_tiling_visualization.ipynb)；
- <strong>修改目标：</strong> 工作副本中的 <code>student_compute.h</code>；
- <strong>完成标志：</strong> 三项接口检查、干净构建、三类非法参数和三组精度用例全部通过，末尾显示 <code>SELF_CHECK PASS</code>。

> 后续单元会写入一份有意保留错误的起始占位代码（下文简称 starter）。starter 自检失败是预期现象。开始修改后，不要重新运行未修改的 starter 单元，否则会覆盖已经完成的代码。

## 1. 实践任务

先确认算子语义、修改范围和完成条件，再开始编码。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>项目</th><th style='text-align: left;'>要求</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'>算子语义</td><td style='text-align: left;'>对两个 <code>float32</code> 一维向量执行逐元素加法：<code>z[i] = x[i] + y[i]</code></td></tr>
    <tr><td style='text-align: left;'>数据切分</td><td style='text-align: left;'>总长度先按 <code>blockDim</code> 均分到各核，每核数据再按 <code>tileCount</code> 均分为真实 GM Tile</td></tr>
    <tr><td style='text-align: left;'>TODO 1</td><td style='text-align: left;'>完成 <code>StudentBlockOffset</code>，计算当前核在完整向量中的起始元素下标</td></tr>
    <tr><td style='text-align: left;'>TODO 2</td><td style='text-align: left;'>完成 <code>StudentTileOffset</code>，计算当前 Tile 在本核数据段中的起始元素下标</td></tr>
    <tr><td style='text-align: left;'>TODO 3</td><td style='text-align: left;'>使用 <code>AscendC::Add</code> 完成两个 LocalTensor 的逐元素向量加法</td></tr>
    <tr><td style='text-align: left;'>允许修改</td><td style='text-align: left;'>仅修改工作副本中的 <code>student_compute.h</code></td></tr>
    <tr><td style='text-align: left;'>不应修改</td><td style='text-align: left;'><code>vector_add_student.asc</code>、<code>CMakeLists.txt</code>、自检脚本和测试用例</td></tr>
    <tr><td style='text-align: left;'>输入约束</td><td style='text-align: left;'>总长度逐级整除，且单个 Tile 的字节数满足 32 Byte 对齐</td></tr>
  </tbody>
</table>

计算阶段应使用 Ascend C Vector API，不要在 Kernel 中编写标量循环逐元素相加。

## 2. 准备独立工作副本

运行下一单元后，实践工程会复制到当前用户专属的临时课程根目录。该根目录由系统临时目录与当前用户标识共同确定（支持 UID 的系统使用 UID，否则使用登录用户名），实际路径会由代码单元打印；后续修改只发生在工作副本中，不会改动仓库里的教学源码。

如果工作副本已经存在，默认继续使用，避免覆盖已经完成的 TODO。需要从头开始时，把下一单元中的 <code>RESET_WORKSPACE</code> 改为 <code>True</code>，运行一次后再改回 <code>False</code>。

In [ ]:
from pathlib import Path
import getpass
import os
import re
import shutil
import subprocess
import sys
import tempfile

RESET_WORKSPACE = False
USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
WORK_ROOT = USER_TEMP_ROOT / '01_vector_add_chapter_practice'
previous_repo = globals().get('REPO_ROOT')
try:
    current = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    current = cached_repo.resolve()

def outside_work_root(path):
    try:
        Path(path).resolve().relative_to(WORK_ROOT)
        return False
    except ValueError:
        return True

search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([current, *current.parents])
REPO_ROOT = next(
    (
        path for path in search_roots
        if outside_work_root(path)
        and (path / 'contrib/tutorials/data_structures_compute').is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库中打开本 Notebook')

os.chdir(REPO_ROOT)

CHAPTER = REPO_ROOT / 'contrib/tutorials/data_structures_compute/01_basic_operations'
if RESET_WORKSPACE and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def copy_missing_tree(source, target):
    target.mkdir(parents=True, exist_ok=True)
    for source_path in source.rglob('*'):
        target_path = target / source_path.relative_to(source)
        if source_path.is_dir():
            target_path.mkdir(parents=True, exist_ok=True)
        elif not target_path.exists():
            target_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, target_path)

copy_missing_tree(CHAPTER / 'src/practice', WORK_ROOT / 'practice')
copy_missing_tree(CHAPTER / 'src/tests', WORK_ROOT / 'tests')
if not (WORK_ROOT / 'check_vector_add.py').is_file():
    shutil.copy2(CHAPTER / 'src/grader/grade_vector_add.py', WORK_ROOT / 'check_vector_add.py')

PROJECT = WORK_ROOT / 'practice'
CHECKER = WORK_ROOT / 'check_vector_add.py'
ANSWER_DIR = CHAPTER / 'answer/01.05_chapter_test'
ANSWER = ANSWER_DIR / 'student_compute.h'
STUDENT_HEADER_FILE = PROJECT / 'student_compute.h'
print('student project:', PROJECT)
print('self-check script:', CHECKER)
print('notebook working directory:', Path.cwd())
print('\npractice source tree:')
subprocess.run(['ls', '-la', str(PROJECT)], check=True)
print('\n--- practice/CMakeLists.txt ---')
subprocess.run(['cat', str(PROJECT / 'CMakeLists.txt')], check=True)

## 3. 读懂三个 TODO 的调用位置

本实践只要求补全头文件，但三个接口分别作用于不同的数据位置。编码前先理清它们的输入、输出和单位。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>TODO</th><th style='text-align: left;'>调用位置</th><th style='text-align: left;'>结果如何使用</th><th style='text-align: left;'>单位与作用域</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>StudentBlockOffset</code></td><td style='text-align: left;'><code>Init</code> 中传入每核长度和当前核编号</td><td style='text-align: left;'>为 x、y、z 的 <code>GlobalTensor</code> 绑定本核负责的 GM 连续区间</td><td style='text-align: left;'>元素下标；相对于完整向量</td></tr>
    <tr><td style='text-align: left;'><code>StudentTileOffset</code></td><td style='text-align: left;'><code>CopyIn</code> 和 <code>CopyOut</code> 中传入 Tile 序号和 Tile 长度</td><td style='text-align: left;'>作为 <code>xGm_[offset]</code>、<code>yGm_[offset]</code>、<code>zGm_[offset]</code> 的核内偏移</td><td style='text-align: left;'>元素下标；相对于本核区间</td></tr>
    <tr><td style='text-align: left;'><code>STUDENT_COMPUTE</code></td><td style='text-align: left;'><code>Compute</code> 中传入输出 Tensor、两个输入 Tensor 和 Tile 长度</td><td style='text-align: left;'>对完整 Tile 执行 <code>z = x + y</code></td><td style='text-align: left;'>数据位于 Local Memory；长度单位为元素数</td></tr>
  </tbody>
</table>

<code>Init</code> 已经让每个核的 <code>GlobalTensor</code> 指向不同 GM 区间，因此 <code>CopyIn</code> 和 <code>CopyOut</code> 中的偏移只描述本核内部的 Tile 位置，不能再次叠加全局核偏移。

实现前先回答三个问题：

1. Block 0、Block 1 和最后一个 Block 的起始下标分别如何计算？
2. 为什么同一 Tile 在 <code>CopyIn</code> 和 <code>CopyOut</code> 中必须使用相同的核内偏移？
3. <code>STUDENT_COMPUTE</code> 收到的三个 Tensor 分别来自哪些队列操作？

## 4. 写入 starter

下面的头文件故意保留三处错误：两个偏移函数都返回 0，计算接口只把 x 复制到 z。它用于帮助你观察“代码能够编译”与“计算结果正确”之间的区别。

<strong>操作顺序：</strong>

1. 第一次运行本单元，写入 starter；
2. 运行后面的快速自检和完整自检，观察最先出现的失败项；
3. 回到本单元补全三个 TODO，再次运行本单元；
4. 重新运行快速自检和完整自检，直到显示 <code>SELF_CHECK PASS</code>。

> 本单元使用 <code>%%writefile</code>，每次运行都会覆盖工作副本中的 <code>student_compute.h</code>。

In [ ]:
%%writefile {STUDENT_HEADER_FILE}
#pragma once

// TODO 1：计算当前核在全局内存中的起始元素下标。
// 提示：每个核连续处理 blockLength 个元素，blockIdx 从 0 开始。
__aicore__ inline uint32_t StudentBlockOffset(uint32_t blockLength, uint32_t blockIdx)
{
    return 0U;
}

// TODO 2：计算当前 Tile 在“本核数据段”中的起始元素下标。
// 提示：progress 是 Tile 序号，tileLength 是单个 Tile 的元素数。
__aicore__ inline uint32_t StudentTileOffset(int32_t progress, uint32_t tileLength)
{
    return 0U;
}

// TODO 3：把占位计算替换为逐元素向量加法 z = x + y。
// 当前占位版本只复制 x，能够编译，但不会通过数值正确性测试。
#define STUDENT_COMPUTE(z, x, y, len) AscendC::Adds((z), (x), 0.0F, (len))

## 5. 完成 TODO 与快速自检

建议按以下顺序完成三个 TODO：

1. <strong>核间偏移：</strong> 手算 Block 0、Block 1 和最后一个 Block 的起点，确认相邻区间连续且不重叠；
2. <strong>核内偏移：</strong> 手算 Tile 0、Tile 1 和最后一个 Tile 的起点，确认最后一个 Tile 不越过本核区间；
3. <strong>Vector 计算：</strong> 确认两个输入 LocalTensor 都参与计算，结果写入输出 Tensor，并处理完整的 <code>len</code> 个元素；
4. <strong>快速检查：</strong> 确认没有保留 <code>return 0U</code> 或 starter 中的 <code>AscendC::Adds</code>。

下一代码单元只检查占位内容是否已经替换，不能证明公式、边界或数值结果正确。最终应以第 6 节的完整自检为准。

In [ ]:
student_header = PROJECT / 'student_compute.h'
student_source = student_header.read_text(encoding='utf-8')
print(student_source)

def function_body(function_name):
    pattern = re.escape(function_name) + r'\s*\([^)]*\)\s*\{(?P<body>.*?)\}'
    match = re.search(pattern, student_source, flags=re.DOTALL)
    return match.group('body') if match else ''

quick_checks = [
    ('TODO 1 已替换 starter 返回值', 'return 0U;' not in function_body('StudentBlockOffset')),
    ('TODO 2 已替换 starter 返回值', 'return 0U;' not in function_body('StudentTileOffset')),
    ('TODO 3 已替换 starter 复制接口', 'AscendC::Adds' not in student_source),
]
print('\nQuick check (not the full validation):')
for description, passed in quick_checks:
    status = 'PASS' if passed else 'CHECK'
    print(f'[{status}] {description}')

## 6. 运行完整自检

完整自检会依次完成：

1. 检查三个学生接口是否使用了题目要求的参数和 <code>AscendC::Add</code>；
2. 在干净目录中配置并编译实践工程；
3. 运行三类非法 Shape，确认 Host 在 Kernel 启动前拒绝错误参数；
4. 运行三组合法 Shape，在 NPU 输出与 CPU Golden 之间进行精度比较。

在环境正常且没有改动其他文件时，starter 的预期结果是：接口检查 0/3、干净构建通过、非法 Shape 3/3；由于前置检查未通过，精度用例暂不运行，末尾显示 <code>SELF_CHECK FAIL</code>。完成三个 TODO 后，目标输出为接口检查 3/3、干净构建通过、非法 Shape 3/3、精度用例 3/3，并在末尾显示 <code>SELF_CHECK PASS</code>。

> 完整自检需要 CANN 开发环境和可用 NPU，运行时间会明显长于普通 Python 单元。

In [ ]:
build_dir = WORK_ROOT / 'build'
check_result = subprocess.run(
    [
        sys.executable, str(CHECKER),
        '--project', str(PROJECT),
        '--build-dir', str(build_dir),
        '--cases', str(WORK_ROOT / 'tests/vector_add_cases.json'),
    ],
    text=True,
    capture_output=True,
)
raw_check_text = (
    (check_result.stdout or '')
    + ('\n[stderr]\n' + check_result.stderr if check_result.stderr else '')
)
visible_lines = []
for line in raw_check_text.splitlines():
    if line.startswith('AUTO_RESULT_'):
        continue
    line = line.replace('[GATE PASS]', '[PASS]').replace('[GATE FAIL]', '[FAIL]')
    line = re.sub(r'(\[(?:PASS|FAIL)\] precision case #\d+)\s+\([^)]*\)', r'\1', line)
    line = line.replace('because a process gate failed', 'because an earlier check failed')
    visible_lines.append(line)
check_text = '\n'.join(visible_lines)
print(check_text)
print('self-check exit code:', check_result.returncode)
print('SELF_CHECK PASS' if check_result.returncode == 0 else 'SELF_CHECK FAIL')

first_failure = next(
    (line for line in check_text.splitlines() if line.startswith('[FAIL]')),
    None,
)
if check_result.returncode == 0:
    print('完整自检通过。')
elif first_failure:
    print('请先处理最早的失败项：', first_failure)
else:
    print('自检未通过，请从 stderr 或最早的异常信息开始排查。')

## 7. 根据第一条失败信息排错

自检输出按执行顺序排列。每次先处理最早出现的失败项，修改后重新运行第 5、6 节。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>现象</th><th style='text-align: left;'>优先检查</th><th style='text-align: left;'>建议做法</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>StudentBlockOffset</code> 检查失败</td><td style='text-align: left;'>返回表达式是否同时使用 <code>blockLength</code> 和 <code>blockIdx</code>；单位是否仍为元素</td><td style='text-align: left;'>手算 Block 0、1 和最后一个 Block 的区间</td></tr>
    <tr><td style='text-align: left;'><code>StudentTileOffset</code> 检查失败</td><td style='text-align: left;'>返回表达式是否同时使用 <code>progress</code> 和 <code>tileLength</code>；是否重复加入全局核偏移</td><td style='text-align: left;'>手算 Tile 0、1 和最后一个 Tile 的核内区间</td></tr>
    <tr><td style='text-align: left;'><code>STUDENT_COMPUTE</code> 检查失败</td><td style='text-align: left;'>是否仍使用 <code>AscendC::Adds</code>，或 API 名称、参数顺序是否错误</td><td style='text-align: left;'>对照 01.03 的 Compute 小节检查三个 Tensor</td></tr>
    <tr><td style='text-align: left;'><code>clean build</code> 失败</td><td style='text-align: left;'>stderr 中最早的编译错误；CANN 环境是否加载</td><td style='text-align: left;'>从第一条编译错误开始处理，不要同时改动多个位置</td></tr>
    <tr><td style='text-align: left;'>非法 Shape 用例失败</td><td style='text-align: left;'>主程序、自检脚本或测试用例是否被改动</td><td style='text-align: left;'>重建工作副本，只修改 <code>student_compute.h</code></td></tr>
    <tr><td style='text-align: left;'>某个 precision case 失败</td><td style='text-align: left;'>多核覆盖、Tile 覆盖或 Vector 计算是否仍有错误</td><td style='text-align: left;'>分别核对多核、多 Tile 和完整长度的数据覆盖</td></tr>
    <tr><td style='text-align: left;'>找不到 NPU 或 ACL 初始化失败</td><td style='text-align: left;'>当前实例是否分配 NPU，CANN 环境是否加载</td><td style='text-align: left;'>检查 Notebook 内核与 <code>npu-smi info</code> 输出</td></tr>
  </tbody>
</table>

不要修改测试用例来绕过失败。多组规模用于确认你的表达式描述了一般性的线性切分，而不是只对默认参数成立。

## 8. 完成检查与答案核对

完成后逐项确认：

- 只修改了 <code>student_compute.h</code> 中的三个 TODO；
- 三项学生接口检查和干净构建均通过；
- 三类非法 Shape 均被 Host 正确拒绝；
- 三组合法 Shape 的精度检查全部通过；
- 自检末尾显示 <code>SELF_CHECK PASS</code>；
- 能解释 Block 偏移与 Tile 偏移的作用域，以及 <code>tileCount</code> 与队列槽位数的区别。

### 独立完成后核对参考答案

参考头文件只用于完成实践后的核对。下面的开关默认为 <code>False</code>，因此顺序运行 Notebook 时不会显示答案。独立完成并通过自检后，再把它改为 <code>True</code> 并单独运行该单元。

In [ ]:
SHOW_REFERENCE_ANSWER = False
if SHOW_REFERENCE_ANSWER:
    subprocess.run(['cat', str(ANSWER_DIR / 'README.md')], check=True)
    subprocess.run(['cat', str(ANSWER)], check=True)
else:
    print('Reference answer is hidden. Finish the practice and pass self-check first.')

## 9. 实践小结

完成本节后，你应能把一维数组的 Block/Tile 划分转化为 GM 元素偏移，并能说明一个 Tile 如何通过队列进入 Local Memory、由 Vector API 完成计算后写回 GM。

如果结果未通过，不要只观察数组开头的少量元素。应从第一条失败信息出发，依次核对核间覆盖、核内 Tile 覆盖和 <code>AscendC::Add</code> 的输入输出，直到所有合法规模均通过 CPU Golden 精度检查。